# SOC L2 Lab — Caza y correlacion

Correlaciona los marcadores `SOC-L2-Pxx` con los incidentes `LAB-Pxx` que crean las reglas analiticas, y traza una linea de tiempo del `RunId`.

> Requisitos: `pip install azure-monitor-query azure-identity pandas`.


In [ ]:
# Instalación de librerías requeridas en Azure ML
%pip install -q azure-identity azure-monitor-query pandas matplotlib


In [ ]:
WORKSPACE_ID = '6cdf51ac-a467-4b2f-9d87-4bd5f69a0bec'
LOOKBACK_HOURS = 24
RUN_ID = ''  # opcional: fija un RunId (p. ej. 20261022nnnnnnnn) para acotar la caza

In [ ]:
from datetime import timedelta
import pandas as pd
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

client = LogsQueryClient(DefaultAzureCredential())

def run_kql(query):
    resp = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(hours=LOOKBACK_HOURS))
    table = resp.tables[0] if resp.status == LogsQueryStatus.SUCCESS else resp.partial_data[0]
    return pd.DataFrame(data=table.rows, columns=table.columns)

## Linea de tiempo unificada de los marcadores SOC-L2


In [ ]:
filtro = f"| where Marcador has '{RUN_ID}'" if RUN_ID else ''
timeline = f'''
let win = SecurityEvent | where CommandLine has 'SOC-L2' or TargetUserName startswith 'soc-l2' or TargetUserName startswith 'SOCp15' | extend Fuente='Windows', Marcador=strcat(TargetUserName,' ',CommandLine) | project TimeGenerated, Fuente, Detalle=Marcador;
let sys = Syslog | where SyslogMessage has 'SOC-L2' or SyslogMessage has 'socp04' | extend Fuente='Linux', Marcador=SyslogMessage | project TimeGenerated, Fuente, Detalle=Marcador;
let app = AppServiceHTTPLogs | where UserAgent startswith 'SOC-L2-Lab' | extend Fuente='AppService', Marcador=strcat(CsMethod,' ',CsUriStem) | project TimeGenerated, Fuente, Detalle=Marcador;
let aks = AKSAuditAdmin | where RequestUri has 'soc-l2-p08' | extend Fuente='AKS', Marcador=RequestUri | project TimeGenerated, Fuente, Detalle=Marcador;
union win, sys, app, aks
{filtro}
| sort by TimeGenerated asc
'''
run_kql(timeline)

## Incidentes LAB-Pxx generados por las reglas analiticas


In [ ]:
run_kql('''
SecurityIncident
| where Title startswith 'LAB-'
| summarize arg_max(TimeGenerated, Severity, Status, Owner, AlertIds) by IncidentNumber, Title
| project IncidentNumber, Title, Severity, Status, Owner
| sort by IncidentNumber desc
''')

## Alertas y entidades asociadas


In [ ]:
run_kql('''
SecurityAlert
| where AlertName startswith 'LAB-'
| project TimeGenerated, AlertName, AlertSeverity, Entities
| sort by TimeGenerated desc
''')

## Conteo por regla para medir cobertura de la demo


In [ ]:
run_kql('''
SecurityAlert
| where AlertName startswith 'LAB-'
| summarize Alertas=count() by AlertName
| sort by Alertas desc
''')